In [ ]:
from pathlib import Path
import openpyxl
from openpyxl.utils import get_column_letter
import os

In [ ]:
!date
jupyter_dir = os.getcwd()
print(f"DIR: {jupyter_dir}")

In [ ]:
xlsx_path = Path("./Subscriber.xlsx")
out_dir = Path("./output")
out_dir.mkdir(parents=True, exist_ok=True)

wb = openpyxl.load_workbook(xlsx_path, data_only=True)

ws_source = wb["SourceAddress"]
ws_black  = wb["BlackList"]
ws_white  = wb["WhiteList"]
ws_cat    = wb["CategoryList"]

In [ ]:
def read_column_list(ws, col: int, header_row: int = 4, start_row: int = 5):
    identifier = ws.cell(row=header_row, column=col).value
    if identifier is None or str(identifier).strip() == "":
        return None, []

    identifier = str(identifier).strip()

    values = []
    r = start_row
    while True:
        v = ws.cell(row=r, column=col).value
        if v is None or str(v).strip() == "":
            break
        values.append(str(v).strip())
        r += 1

    return identifier, values

def append_section_yaml(path: Path, key: str, values: list[str]):
    with path.open("a", encoding="utf-8") as f:
        if path.stat().st_size > 0:
            f.write("\n")

        if not values:
            f.write(f"{key}: []\n")
            return

        f.write(f"{key}:\n")
        for v in values:
            f.write(f" - {v}\n")

col = 3  # C列スタート
print(f"Processing started: reading from column {get_column_letter(col)}")

while True:
    identifier, source_addresses = read_column_list(ws_source, col)
    if identifier is None:
        break

    out_path = out_dir / f"{identifier}.yaml"
    out_path.write_text("", encoding="utf-8")  # 毎回リセット

    append_section_yaml(out_path, "souceAddress", source_addresses)

    # BlackList
    black_identifier, black_values = read_column_list(ws_black, col)
    if black_identifier is None:
        print(f"[WARN] BlackList: identifier not found at column {get_column_letter(col)} for '{identifier}'")

    # WhiteList
    white_identifier, white_values = read_column_list(ws_white, col)
    if white_identifier is None:
        print(f"[WARN] WhiteList: identifier not found at column {get_column_letter(col)} for '{identifier}'")

    # CategoryList
    cat_identifier, cat_values = read_column_list(ws_cat, col)
    if cat_identifier is None:
        print(f"[WARN] CategoryList: identifier not found at column {get_column_letter(col)} for '{identifier}'")

    append_section_yaml(out_path, "blackList", black_values)
    append_section_yaml(out_path, "whiteList", white_values)
    append_section_yaml(out_path, "categoryList", cat_values)

    print(f"written: {out_path}")
    col += 1

print("All processing completed.")